# Review knowledge questions

**Goal:** Inspect AI revisions, then accept, reject or correct questions through the human-review form.

Run cells from top to bottom. Default cells work offline; model execution is an explicit opt-in and writes only to ignored `outputs/`.

## 1. Set up paths

Find the repository and load the analysis helpers.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/final").is_dir(), "Run from the repository or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from sleepinn_study.io import read_json, read_jsonl, output_directory
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})


## 2. Read the AI revision instructions

This is the AI editorial/source-fidelity step. It is separate from the subsequent sleep-expert review. Public data omit verbatim evidence excerpts; a reviewer should open the cited licensed source.

In [ ]:
print((ROOT / "data/config/databank/ai_revision_instructions.md").read_text(encoding="utf-8"))
before = read_jsonl(ROOT / "data/candidates/knowledge.jsonl")
reviewed = read_jsonl(ROOT / "data/reviewed/knowledge.jsonl")
display(pd.DataFrame(reviewed).item_type.value_counts().rename("after AI revision"))
# For a new AI revision, load the saved review response and apply its explicit patches.
from sleepinn_study.workflow import apply_ai_reviews
APPLY_NEW_AI_REVIEW = False
if APPLY_NEW_AI_REVIEW:
    response = read_json(ROOT / "outputs/authoring/ai_review_response.json")
    reviewed = apply_ai_reviews(before, response["reviews"])


## 3. Open the human-review form

The author reports that expert review of the released bank was completed without further content changes. This form records an actual new review; it starts with no prefilled acceptances. Enter a reviewer identifier, check the source, then accept, reject or save a correction. Accepted variants and the answer rationale are editable.

In [ ]:
from sleepinn_study.review import QuestionReviewer
reviewer = QuestionReviewer(reviewed, ROOT / "outputs/human_review/knowledge")
display(reviewer.widget)

## 4. Export only after all decisions

Use “Export final reviewed bank” in the form when every question has a decision. Rejected questions are excluded; the export manifest records counts and hashes. Combine completed knowledge and clinical exports in notebook 03.

In [ ]:
print("Decisions recorded:", len(reviewer.session.latest), "/", len(reviewed))
print("Current released bank:", ROOT / "data/final/knowledge.jsonl")
print("New review exports:", ROOT / "outputs/human_review/knowledge/exports")